Install Packages and Import Libraries

In [ ]:
# %pip install -q -r ../requirements.txt

In [1]:
# Import required libraries and modules for zero-shot sampling and few-shot training.
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from huggingface_hub import HfApi, login
from IPython.display import display

warnings.filterwarnings('ignore')

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.local.classification.vlm.models import (load_openai_clip, load_biomedclip, load_unimedclip, make_openai_clip_loader, make_biomedclip_loader, make_unimedclip_loader)
from src.local.classification.vlm.helpers import (load_busi_splits, save_results_to_csv, encode_images_batch)
from src.local.classification.vlm.zero_shot import ZeroShotEvaluator
from src.local.prompting.prompt_registry import PROMPT_REGISTRY
from src.local.classification.vlm.few_shot_lp import (LinearProbeConfig, make_kshot_indices, run_linear_probe_experiments)
from src.local.classification.vlm.few_shot_lora import get_args, run_kshot_experiments

c:\Users\mason\Desktop\busi-vlm-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authenticate Hugging Face and Select Device

In [2]:
load_dotenv(project_root / '.env', override=True)

huggingface_token = os.getenv('huggingface_token')
api = HfApi(token=huggingface_token) if huggingface_token else None

if huggingface_token:
    login(token=huggingface_token, add_to_git_credential=False)
    print('HuggingFace authenticated as -', api.whoami()['name'])
else:
    print('No huggingface_token found.')

HuggingFace authenticated as - masonezra34


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device -', device)

if device == 'cuda':
    print(torch.cuda.get_device_name(0))

device - cuda
NVIDIA GeForce RTX 5060 Laptop GPU


Define BUSI Classes and Prompt Labels

In [4]:
# Setup BUSI classes and prompts.
busi_classes = ['benign', 'malignant', 'normal']

class_mapping = {
    'benign': 'benign tumor',
    'malignant': 'malignant tumor',
    'normal': 'normal scan'
}

class_names_for_prompts = [class_mapping[c] for c in busi_classes]

print(f'BUSI classes - {busi_classes}')
print(f'prompt registry classes - {class_names_for_prompts}')
print(f'prompts per class:')

for class_name in class_names_for_prompts:
    print(f'  {class_name}: {len(PROMPT_REGISTRY[class_name])} prompts')

BUSI classes - ['benign', 'malignant', 'normal']
prompt registry classes - ['benign tumor', 'malignant tumor', 'normal scan']
prompts per class:
  benign tumor: 8 prompts
  malignant tumor: 8 prompts
  normal scan: 8 prompts


Load BUSI Train, Validation, and Test Splits

In [5]:
# Load BUSI train/val/test splits.
train_df, val_df, test_df = load_busi_splits(project_root, busi_classes)

print(f'\ndataset splits:')
print(f'* train - {len(train_df)}')
print(f'* validation - {len(val_df)}')
print(f'* test - {len(test_df)}')
print(f'\ntest distribution:')
print(test_df['label'].value_counts())


dataset splits:
* train - 437
* validation - 93
* test - 95

test distribution:
label
benign       51
malignant    29
normal       15
Name: count, dtype: int64


 Zero-Shot Evaluation using OpenAI CLIP ViT-B/16

In [ ]:
# Load OpenAI CLIP ViT-B/16.
openai_model, openai_preprocess, openai_tokenizer = load_openai_clip(device=device)

# Create evaluator.
openai_evaluator = ZeroShotEvaluator(openai_model, openai_preprocess, openai_tokenizer, PROMPT_REGISTRY, class_names_for_prompts, device=device)

# Build text embeddings.
openai_evaluator.build_text_embeddings()

# Evaluate on test set.
openai_metrics, _, _ = openai_evaluator.evaluate(test_df, batch_size=32, description='encoding test images (OpenAI CLIP)')

# Print results.
openai_evaluator.print_results(openai_metrics, 'OpenAI CLIP ViT-B/16')

# Save results.
results_path = project_root / 'results' / 'zero_shot_results.csv'
results_path.parent.mkdir(exist_ok=True)
save_results_to_csv(openai_metrics, results_path, 'openai_clip_vit_b16', append=False)
print(f'\nresults saved to - {results_path}')

benign tumor - torch.Size([8, 512])
malignant tumor - torch.Size([8, 512])
normal scan - torch.Size([8, 512])


encoding test images (OpenAI CLIP): 100%|██████████| 3/3 [00:01<00:00,  1.88it/s]


Zero-Shot Results: OpenAI CLIP ViT-B/16
accuracy - 0.5895
balanced Accuracy - 0.4585
macro F1 - 0.4589
AUC - 0.7102

per-class F1 -
  benign: 0.7132
  malignant: 0.3158
  normal: 0.3478

results saved to - c:\Users\mason\Desktop\busi-vlm-project\results\zero_shot_results.csv


Zero-Shot Evaluation using BiomedCLIP ViT-B/16

In [ ]:
# Load BiomedCLIP ViT-B/16.
biomedclip_model, _, biomedclip_preprocess, biomedclip_tokenizer = load_biomedclip(device=device)

# Create evaluator.
biomedclip_evaluator = ZeroShotEvaluator(biomedclip_model, biomedclip_preprocess, biomedclip_tokenizer, PROMPT_REGISTRY, class_names_for_prompts, device=device)

# Build text embeddings.
biomedclip_evaluator.build_text_embeddings()

# Evaluate on test set.
biomedclip_metrics, _, _ = biomedclip_evaluator.evaluate(test_df, batch_size=32, description='encoding test images (BiomedCLIP)')

# Print results.
biomedclip_evaluator.print_results(biomedclip_metrics, 'BiomedCLIP ViT-B/16')

# Save results.
save_results_to_csv(biomedclip_metrics, results_path, 'biomedclip_vit_b16', append=True)

print(f'\nresults saved to - {results_path}')

benign tumor - torch.Size([8, 512])
malignant tumor - torch.Size([8, 512])
normal scan - torch.Size([8, 512])


encoding test images (BiomedCLIP): 100%|██████████| 3/3 [00:01<00:00,  2.29it/s]


Zero-Shot Results: BiomedCLIP ViT-B/16
accuracy - 0.4105
balanced Accuracy - 0.4771
macro F1 - 0.3786
AUC - 0.7465

per-class F1 -
  benign: 0.1786
  malignant: 0.5225
  normal: 0.4348

results saved to - c:\Users\mason\Desktop\busi-vlm-project\results\zero_shot_results.csv


Zero-Shot Evaluation using UniMed-CLIP ViT-B/16

In [ ]:
# Load UniMed-CLIP ViT-B/16.
unimedclip_model, unimedclip_preprocess, unimedclip_tokenizer = load_unimedclip(device=device, project_root=project_root)

# Create evaluator.
unimedclip_evaluator = ZeroShotEvaluator(unimedclip_model, unimedclip_preprocess, unimedclip_tokenizer, PROMPT_REGISTRY, class_names_for_prompts, device=device)

# Build text embeddings.
unimedclip_evaluator.build_text_embeddings()

# Evaluate on test set.
unimedclip_metrics, _, _ = unimedclip_evaluator.evaluate(test_df, batch_size=32, description='encoding test images (UniMed-CLIP)')

# Print results.
unimedclip_evaluator.print_results(unimedclip_metrics, 'UniMed-CLIP ViT-B/16')

# Save results.
save_results_to_csv(unimedclip_metrics, results_path, 'unimedclip', append=True)
print(f'\nresults saved to - {results_path}')

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 106041.82it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can b

benign tumor - torch.Size([8, 512])
malignant tumor - torch.Size([8, 512])
normal scan - torch.Size([8, 512])


encoding test images (UniMed-CLIP): 100%|██████████| 3/3 [00:01<00:00,  2.22it/s]


Zero-Shot Results: UniMed-CLIP ViT-B/16
accuracy - 0.3053
balanced Accuracy - 0.3968
macro F1 - 0.3042
AUC - 0.5876

per-class F1 -
  benign: 0.3235
  malignant: 0.2917
  normal: 0.2973

results saved to - c:\Users\mason\Desktop\busi-vlm-project\results\zero_shot_results.csv


Compare Zero-Shot Results

In [9]:
# Display zero-shot comparison.
zero_shot_results = pd.read_csv(results_path)
display(zero_shot_results)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,auc,benign_precision,benign_recall,benign_f1,malignant_precision,malignant_recall,malignant_f1,normal_precision,normal_recall,normal_f1
0,openai_clip_vit_b16,0.589474,0.458508,0.458931,0.534183,0.710239,0.589744,0.901961,0.713178,0.666667,0.206897,0.315789,0.500000,0.266667,0.347826
1,biomedclip_vit_b16,0.410526,0.477124,0.378626,0.324021,0.746545,1.000000,0.098039,0.178571,0.353659,1.000000,0.522523,0.625000,0.333333,0.434783
2,unimedclip,0.305263,0.396800,0.304164,0.309661,0.587574,0.647059,0.215686,0.323529,0.368421,0.241379,0.291667,0.186441,0.733333,0.297297


Extract Frozen Image Features for Linear Probing

In [10]:
feature_batch_size = 32

model_specs = [
    {
        'model_name': 'openai_clip_vit_b16',
        'model': openai_model,
        'preprocess': openai_preprocess,
    },
    {
        'model_name': 'biomedclip_vit_b16',
        'model': biomedclip_model,
        'preprocess': biomedclip_preprocess,
    },
    {
        'model_name': 'unimedclip',
        'model': unimedclip_model,
        'preprocess': unimedclip_preprocess,
    },
]

feature_sets = {}

for spec in model_specs:
    model_name = spec['model_name']
    model = spec['model']
    preprocess = spec['preprocess']

    print(f'encoding frozen features - {model_name}')

    model.eval()

    feature_sets[model_name] = {
        'train': encode_images_batch(
            model,
            preprocess,
            train_df,
            device=device,
            batch_size=feature_batch_size,
            description=f'{model_name} train',
        ),
        'val': encode_images_batch(
            model,
            preprocess,
            val_df,
            device=device,
            batch_size=feature_batch_size,
            description=f'{model_name} val',
        ),
        'test': encode_images_batch(
            model,
            preprocess,
            test_df,
            device=device,
            batch_size=feature_batch_size,
            description=f'{model_name} test',
        ),
    }

    print(
        f"{model_name} features - "
        f"train {feature_sets[model_name]['train'].shape}, "
        f"val {feature_sets[model_name]['val'].shape}, "
        f"test {feature_sets[model_name]['test'].shape}"
    )

train_labels = train_df['label_index'].values
val_labels = val_df['label_index'].values
test_labels = test_df['label_index'].values

encoding frozen features - openai_clip_vit_b16


openai_clip_vit_b16 test: 100%|██████████| 3/3 [00:01<00:00,  2.24it/s]


openai_clip_vit_b16 features - train torch.Size([437, 512]), val torch.Size([93, 512]), test torch.Size([95, 512])
encoding frozen features - biomedclip_vit_b16


biomedclip_vit_b16 test: 100%|██████████| 3/3 [00:01<00:00,  2.37it/s]


biomedclip_vit_b16 features - train torch.Size([437, 512]), val torch.Size([93, 512]), test torch.Size([95, 512])
encoding frozen features - unimedclip


unimedclip test: 100%|██████████| 3/3 [00:01<00:00,  2.28it/s]

unimedclip features - train torch.Size([437, 512]), val torch.Size([93, 512]), test torch.Size([95, 512])


Set Up Shared Few-Shot Support Sets

In [11]:
# Few-shot setup.
shots_per_class = [1, 2, 4, 8, 16, 32]
seeds = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

train_labels = train_df['label_index'].values
val_labels = val_df['label_index'].values
test_labels = test_df['label_index'].values

shared_support_indices = make_kshot_indices(
    train_df,
    label_col='label_index',
    shots_per_class=shots_per_class,
    seeds=seeds,
)

Few-Shot Linear Probe Baseline

In [12]:
# Few-shot linear probe. The encoder is frozen and the linear classifier is trained on pre-extracted image features.
# We use a wider C search list from 0.000001 to 1000000.0 to tune the LogisticRegression regularisation strength for each few-shot setting.
probe_config = LinearProbeConfig(
    max_iter = 5000, 
    class_weight = 'balanced', 
    c_values = (0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0), 
    selection_metric = 'macro_f1'
)

all_results = []
all_aggregates = []

for spec in model_specs:
    model_name = spec['model_name']
    features = feature_sets[model_name]

    print(f'few-shot linear probe - {model_name}')

    results_df, aggregate_df = run_linear_probe_experiments(
        model_name=model_name,
        train_features=features["train"],
        train_labels=train_labels,
        val_features=features["val"],
        val_labels=val_labels,
        test_features=features["test"],
        test_labels=test_labels,
        class_names=busi_classes,
        ratios=shots_per_class,
        seeds=seeds,
        device=device,
        config=probe_config,
        verbose=False,
        kshot_indices=shared_support_indices
    )

    all_results.append(results_df)
    all_aggregates.append(aggregate_df)

fewshot_results = pd.concat(all_results, ignore_index=True)
fewshot_aggregate = pd.concat(all_aggregates, ignore_index=True)

summary_columns = [
    'model',
    'shots_per_class',
    'n_train_samples_mean',
    'accuracy_mean',
    'accuracy_std',
    'balanced_accuracy_mean',
    'balanced_accuracy_std',
    'macro_f1_mean',
    'macro_f1_std',
    'auc_mean',
    'auc_std'
]

display(fewshot_aggregate[summary_columns])

fewshot_dir = project_root / 'results' / 'fewshot_linear_probe'
fewshot_dir.mkdir(parents=True, exist_ok=True)

fewshot_results_path = fewshot_dir / 'fewshot_linear_probe_all_runs.csv'
fewshot_aggregate_path = fewshot_dir / 'fewshot_linear_probe_aggregate.csv'

fewshot_results.to_csv(fewshot_results_path, index=False)
fewshot_aggregate.to_csv(fewshot_aggregate_path, index=False)

print(f'few-shot runs saved to - {fewshot_results_path}')
print(f'few-shot aggregate saved to - {fewshot_aggregate_path}')

few-shot linear probe - openai_clip_vit_b16
model=openai_clip_vit_b16 shots=1 seed=1 | n=3 | c=1000.0 | val_f1=0.3106 | test_acc=0.2842 | test_f1=0.2671 | test_auc=0.6262
model=openai_clip_vit_b16 shots=1 seed=2 | n=3 | c=0.1 | val_f1=0.3912 | test_acc=0.4632 | test_f1=0.3942 | test_auc=0.5202
model=openai_clip_vit_b16 shots=1 seed=3 | n=3 | c=10.0 | val_f1=0.3077 | test_acc=0.2316 | test_f1=0.2284 | test_auc=0.6065
model=openai_clip_vit_b16 shots=1 seed=4 | n=3 | c=1.0 | val_f1=0.3474 | test_acc=0.4842 | test_f1=0.2757 | test_auc=0.5258
model=openai_clip_vit_b16 shots=1 seed=5 | n=3 | c=1000.0 | val_f1=0.3395 | test_acc=0.5263 | test_f1=0.4613 | test_auc=0.5714
model=openai_clip_vit_b16 shots=1 seed=6 | n=3 | c=1.0 | val_f1=0.3955 | test_acc=0.4947 | test_f1=0.4332 | test_auc=0.6205
model=openai_clip_vit_b16 shots=1 seed=7 | n=3 | c=0.1 | val_f1=0.4570 | test_acc=0.4842 | test_f1=0.4890 | test_auc=0.7358
model=openai_clip_vit_b16 shots=1 seed=8 | n=3 | c=0.1 | val_f1=0.4131 | test_acc

,model,shots_per_class,n_train_samples_mean,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,auc_mean,auc_std
0,openai_clip_vit_b16,1,3.0,0.417895,0.097371,0.425977,0.081931,0.363616,0.091253,0.592459,0.062857
1,openai_clip_vit_b16,2,6.0,0.396842,0.071229,0.414609,0.059215,0.353611,0.062737,0.589182,0.058942
2,openai_clip_vit_b16,4,12.0,0.509474,0.098771,0.532955,0.073937,0.486383,0.094018,0.704749,0.063212
3,openai_clip_vit_b16,8,24.0,0.506316,0.080388,0.554109,0.070501,0.491295,0.076941,0.725962,0.065789
4,openai_clip_vit_b16,16,48.0,0.580000,0.049733,0.621758,0.047743,0.567467,0.049637,0.792098,0.032243
5,openai_clip_vit_b16,32,96.0,0.627368,0.050164,0.663899,0.044015,0.614018,0.048308,0.819875,0.034551
6,biomedclip_vit_b16,1,3.0,0.437895,0.094176,0.455722,0.109757,0.396557,0.086791,0.663741,0.113909
7,biomedclip_vit_b16,2,6.0,0.527368,0.142220,0.573960,0.135318,0.514834,0.143334,0.735255,0.110464
8,biomedclip_vit_b16,4,12.0,0.645263,0.043839,0.669475,0.041019,0.628804,0.039956,0.818208,0.028601
9,biomedclip_vit_b16,8,24.0,0.656842,0.043599,0.692351,0.041935,0.647901,0.039785,0.825581,0.022897


few-shot runs saved to - c:\Users\mason\Desktop\busi-vlm-project\results\fewshot_linear_probe\fewshot_linear_probe_all_runs.csv
few-shot aggregate saved to - c:\Users\mason\Desktop\busi-vlm-project\results\fewshot_linear_probe\fewshot_linear_probe_aggregate.csv


Shared LoRA Setup

In [13]:
# Use BUSI class names here, not prompt-mapped zero-shot names.
lora_class_names = busi_classes

lora_base_save_dir = project_root/'results'/'few_shot_lora'
lora_base_save_dir.mkdir(parents=True, exist_ok=True)

Few-Shot LoRA Fine-Tuning using OpenAI CLIP ViT-B/16

In [14]:
# Run LoRA training for OpenAI CLIP.
args = get_args([])
args.console_log = False
args.show_progress = True
args.log_epochs = False
args.log_train_metrics = False

args.model_name = 'openai_clip_vit_b16'
args.device = device
args.num_classes = len(busi_classes)
args.encoder = 'vision'

# Training knobs.
args.epochs = 100
args.batch_size = 8
args.accumulation_steps = 4
args.patience = 18
args.grad_clip = 1.0

# Optimizer knobs.
args.lr = 1e-4
args.head_lr = 1e-3
args.lora_lr = 1e-4
args.lr_min = 1e-7
args.head_weight_decay = 1e-3
args.lora_weight_decay = 1e-5

# LoRA knobs.
args.lora_layers = None
args.lora_rank = 16
args.lora_alpha = 32
args.lora_dropout = 0.1

# Target q, k, v, and output projection for a fair comparison with BiomedCLIP qkv + proj.
args.clip_lora_targets = ['q', 'k', 'v', 'o']

# Use eval preprocessing during LoRA training for a controlled few-shot comparison.
args.use_eval_preprocess_for_train = False

save_dir = lora_base_save_dir / args.model_name

openai_lora_results_df, openai_lora_summary_df = run_kshot_experiments(
    args=args,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    class_names=lora_class_names,
    ks=shots_per_class,
    seeds=seeds,
    save_dir=save_dir,
    support_indices=shared_support_indices,
)

openai_lora_summary_df


LoRA few-shot: model=openai_clip | k=[1, 2, 4, 8, 16, 32] | seeds=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] | rank=16 | alpha=32 | dropout=0.1 | layers=None
injected lora adapters to 12 layers (clip vision encoder)
trainable params: 1,179,648 / 150,800,385

LoRA model check: openai_clip
  trainable vision tensors: 96
  vision trainable params: 1,179,648 / 150,800,385
  classifier trainable params: 1,539
  total trainable params: 1,181,187



openai_clip k=1 seed=1:  23%|██▎       | 23/100 [00:40<02:17,  1.78s/it, best=0.4655, head_lr=8.6e-04, lora_lr=8.6e-05, loss=0.7079, val_f1=0.4088]


done model=openai_clip k=1 seed=1 best_epoch=6 best_val_f1=0.4655 test_acc=0.2526 test_f1=0.2627 test_auc=0.4823


openai_clip k=1 seed=2:  33%|███▎      | 33/100 [01:02<02:06,  1.89s/it, best=0.4514, head_lr=7.4e-04, lora_lr=7.4e-05, loss=0.5502, val_f1=0.4004]


done model=openai_clip k=1 seed=2 best_epoch=16 best_val_f1=0.4514 test_acc=0.4421 test_f1=0.4210 test_auc=0.5695


...

openai_clip k=32 seed=9:  37%|███▋      | 37/100 [02:30<04:15,  4.06s/it, best=0.7524, head_lr=6.8e-04, lora_lr=6.8e-05, loss=0.1494, val_f1=0.7173]


done model=openai_clip k=32 seed=9 best_epoch=20 best_val_f1=0.7524 test_acc=0.8105 test_f1=0.7986 test_auc=0.8983


openai_clip k=32 seed=10:  44%|████▍     | 44/100 [02:55<03:43,  3.99s/it, best=0.7859, head_lr=5.8e-04, lora_lr=5.8e-05, loss=0.1079, val_f1=0.7697]


done model=openai_clip k=32 seed=10 best_epoch=27 best_val_f1=0.7859 test_acc=0.7158 test_f1=0.7113 test_auc=0.8912

LoRA summary:
 k  best_epoch_mean  best_val_macro_f1_mean  best_val_macro_f1_std  test_accuracy_mean  test_accuracy_std  test_macro_f1_mean  test_macro_f1_std  test_auc_mean  test_auc_std
 1             19.4                0.415706               0.070930            0.409474           0.093025            0.359189           0.089457       0.584313      0.079520
 2             19.0                0.420040               0.078567            0.382105           0.093369            0.333550           0.085836       0.600250      0.093395
 4             24.2                0.561091               0.051646            0.529474           0.103142            0.491155           0.087925       0.729743      0.054109
 8             35.3                0.624828               0.067207            0.607368           0.055711            0.589619           0.066672       0.790719      0.055453

,k,best_epoch_mean,best_val_macro_f1_mean,best_val_macro_f1_std,test_accuracy_mean,test_accuracy_std,test_macro_f1_mean,test_macro_f1_std,test_auc_mean,test_auc_std
0,1,19.4,0.415706,0.070930,0.409474,0.093025,0.359189,0.089457,0.584313,0.079520
1,2,19.0,0.420040,0.078567,0.382105,0.093369,0.333550,0.085836,0.600250,0.093395
2,4,24.2,0.561091,0.051646,0.529474,0.103142,0.491155,0.087925,0.729743,0.054109
3,8,35.3,0.624828,0.067207,0.607368,0.055711,0.589619,0.066672,0.790719,0.055453
4,16,30.3,0.716215,0.044689,0.695789,0.065915,0.682602,0.066368,0.846113,0.037890
5,32,26.9,0.783172,0.033394,0.743158,0.033361,0.735095,0.037416,0.886924,0.019947


Few-Shot LoRA Fine-Tuning using BiomedCLIP ViT-B/16

In [15]:
# Run LoRA training for BiomedCLIP.
args = get_args([])
args.console_log = False
args.show_progress = True
args.log_epochs = False
args.log_train_metrics = False

args.model_name = 'biomedclip_vit_b16'
args.device = device
args.num_classes = len(busi_classes)
args.encoder = 'vision'

# Training knobs.
args.epochs = 100
args.batch_size = 8
args.accumulation_steps = 4
args.patience = 18
args.grad_clip = 1.0

# Optimizer knobs.
args.lr = 1e-4
args.head_lr = 1e-3
args.lora_lr = 1e-4
args.lr_min = 1e-7
args.head_weight_decay = 1e-3
args.lora_weight_decay = 1e-5

# LoRA knobs.
args.lora_layers = None
args.lora_rank = 16
args.lora_alpha = 32
args.lora_dropout = 0.1

# Target fused qkv and output projection for fair comparison with OpenAI CLIP q + k + v + o.
args.biomed_lora_targets = ['qkv', 'proj']

# Use eval preprocessing during LoRA training for a controlled few-shot comparison.
args.use_eval_preprocess_for_train = False

save_dir = lora_base_save_dir / args.model_name

biomedclip_lora_results_df, biomedclip_lora_summary_df = run_kshot_experiments(
    args=args,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    class_names=lora_class_names,
    ks=shots_per_class,
    seeds=seeds,
    save_dir=save_dir,
    support_indices=shared_support_indices,
)

biomedclip_lora_summary_df


LoRA few-shot: model=biomedclip | k=[1, 2, 4, 8, 16, 32] | seeds=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] | rank=16 | alpha=32 | dropout=0.1 | layers=None
trainable params: 884,736 || all params: 196,787,457 || trainable%: 0.4496

LoRA model check: biomedclip
  trainable vision tensors: 48
  vision trainable params: 884,736 / 196,787,457
  classifier trainable params: 1,539
  total trainable params: 886,275



biomedclip k=1 seed=1:  73%|███████▎  | 73/100 [01:55<00:42,  1.59s/it, best=0.4684, head_lr=1.6e-04, lora_lr=1.6e-05, loss=0.4240, val_f1=0.3808]


done model=biomedclip k=1 seed=1 best_epoch=56 best_val_f1=0.4684 test_acc=0.5263 test_f1=0.3749 test_auc=0.5962


biomedclip k=1 seed=2:  72%|███████▏  | 72/100 [01:52<00:43,  1.56s/it, best=0.3846, head_lr=1.7e-04, lora_lr=1.7e-05, loss=0.4527, val_f1=0.3797]


done model=biomedclip k=1 seed=2 best_epoch=55 best_val_f1=0.3846 test_acc=0.4211 test_f1=0.3061 test_auc=0.5484


...

biomedclip k=32 seed=9:  54%|█████▍    | 54/100 [03:41<03:09,  4.11s/it, best=0.7692, head_lr=4.2e-04, lora_lr=4.2e-05, loss=0.1830, val_f1=0.7295]


done model=biomedclip k=32 seed=9 best_epoch=37 best_val_f1=0.7692 test_acc=0.7053 test_f1=0.7143 test_auc=0.8916


biomedclip k=32 seed=10:  31%|███       | 31/100 [02:07<04:44,  4.13s/it, best=0.7727, head_lr=7.7e-04, lora_lr=7.7e-05, loss=0.4627, val_f1=0.7353]


done model=biomedclip k=32 seed=10 best_epoch=14 best_val_f1=0.7727 test_acc=0.7263 test_f1=0.7377 test_auc=0.8831

LoRA summary:
 k  best_epoch_mean  best_val_macro_f1_mean  best_val_macro_f1_std  test_accuracy_mean  test_accuracy_std  test_macro_f1_mean  test_macro_f1_std  test_auc_mean  test_auc_std
 1             31.7                0.472345               0.098654            0.480000           0.080808            0.432656           0.079574       0.669602      0.087496
 2             29.2                0.542696               0.148860            0.530526           0.137801            0.514386           0.136030       0.718877      0.130848
 4             37.6                0.637319               0.141963            0.612632           0.126977            0.596512           0.144080       0.803095      0.099336
 8             32.3                0.709948               0.050411            0.676842           0.040022            0.665535           0.043271       0.847676      0.025402


,k,best_epoch_mean,best_val_macro_f1_mean,best_val_macro_f1_std,test_accuracy_mean,test_accuracy_std,test_macro_f1_mean,test_macro_f1_std,test_auc_mean,test_auc_std
0,1,31.7,0.472345,0.098654,0.480000,0.080808,0.432656,0.079574,0.669602,0.087496
1,2,29.2,0.542696,0.148860,0.530526,0.137801,0.514386,0.136030,0.718877,0.130848
2,4,37.6,0.637319,0.141963,0.612632,0.126977,0.596512,0.144080,0.803095,0.099336
3,8,32.3,0.709948,0.050411,0.676842,0.040022,0.665535,0.043271,0.847676,0.025402
4,16,23.5,0.762942,0.024306,0.729474,0.055044,0.729225,0.053428,0.878630,0.029721
5,32,32.5,0.790002,0.028152,0.744211,0.036481,0.748729,0.038238,0.896770,0.015492


Few-Shot LoRA Fine-Tuning using UniMed-CLIP ViT-B/16

In [16]:
# Run LoRA training for UniMed-CLIP.
args = get_args([])
args.console_log = False
args.show_progress = True
args.log_epochs = False
args.log_train_metrics = False

args.model_name = 'unimedclip'
args.device = device
args.project_root = str(project_root)
args.num_classes = len(busi_classes)
args.encoder = 'vision'

# Training knobs.
args.epochs = 100
args.batch_size = 8
args.accumulation_steps = 4
args.patience = 18
args.grad_clip = 1.0

# Optimizer knobs.
# The optimizer uses head_lr and lora_lr, not args.lr.
args.lr = 1e-4
args.head_lr = 1e-3
args.lora_lr = 1e-4
args.lr_min = 1e-7
args.head_weight_decay = 1e-3
args.lora_weight_decay = 1e-5

# LoRA knobs.
args.lora_layers = None
args.lora_rank = 16
args.lora_alpha = 32
args.lora_dropout = 0.1

# Target q, k, v, and output projection for fair comparison with BiomedCLIP qkv + proj.
args.clip_lora_targets = ['q', 'k', 'v', 'o']

# Use eval preprocessing during LoRA training for a controlled few-shot comparison.
args.use_eval_preprocess_for_train = False

save_dir = lora_base_save_dir / args.model_name

unimedclip_lora_results_df, unimedclip_lora_summary_df = run_kshot_experiments(
    args=args,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    class_names=lora_class_names,
    ks=shots_per_class,
    seeds=seeds,
    save_dir=save_dir,
    support_indices=shared_support_indices,
)

unimedclip_lora_summary_df


LoRA few-shot: model=unimedclip | k=[1, 2, 4, 8, 16, 32] | seeds=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] | rank=16 | alpha=32 | dropout=0.1 | layers=None


[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you 

injected lora adapters to 12 layers (clip vision encoder)
trainable params: 1,179,648 / 197,083,137

LoRA model check: unimedclip
  trainable vision tensors: 96
  vision trainable params: 1,179,648 / 197,083,137
  classifier trainable params: 1,539
  total trainable params: 1,181,187



unimedclip k=1 seed=1:  42%|████▏     | 42/100 [01:16<01:45,  1.83s/it, best=0.4859, head_lr=6.1e-04, lora_lr=6.1e-05, loss=0.6727, val_f1=0.4839]


done model=unimedclip k=1 seed=1 best_epoch=25 best_val_f1=0.4859 test_acc=0.4211 test_f1=0.3811 test_auc=0.5597


...

[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you 

done model=unimedclip k=32 seed=9 best_epoch=45 best_val_f1=0.6660 test_acc=0.7263 test_f1=0.7086 test_auc=0.8829


[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you 

done model=unimedclip k=32 seed=10 best_epoch=68 best_val_f1=0.7267 test_acc=0.7368 test_f1=0.7416 test_auc=0.9043

LoRA summary:
 k  best_epoch_mean  best_val_macro_f1_mean  best_val_macro_f1_std  test_accuracy_mean  test_accuracy_std  test_macro_f1_mean  test_macro_f1_std  test_auc_mean  test_auc_std
 1             23.3                0.388025               0.068330            0.373684           0.090176            0.327534           0.053376       0.560920      0.054279
 2             16.8                0.417800               0.089358            0.369474           0.075165            0.329347           0.056603       0.558429      0.066523
 4             41.2                0.528493               0.119565            0.546316           0.085725            0.501206           0.094453       0.723770      0.073408
 8             36.6                0.487357               0.078601            0.481053           0.096482            0.446271           0.100801       0.668345      0.094073


,k,best_epoch_mean,best_val_macro_f1_mean,best_val_macro_f1_std,test_accuracy_mean,test_accuracy_std,test_macro_f1_mean,test_macro_f1_std,test_auc_mean,test_auc_std
0,1,23.3,0.388025,0.068330,0.373684,0.090176,0.327534,0.053376,0.560920,0.054279
1,2,16.8,0.417800,0.089358,0.369474,0.075165,0.329347,0.056603,0.558429,0.066523
2,4,41.2,0.528493,0.119565,0.546316,0.085725,0.501206,0.094453,0.723770,0.073408
3,8,36.6,0.487357,0.078601,0.481053,0.096482,0.446271,0.100801,0.668345,0.094073
4,16,33.1,0.599337,0.099162,0.594737,0.069514,0.557264,0.090337,0.793937,0.065648
5,32,35.1,0.676532,0.097404,0.638947,0.118059,0.631857,0.122407,0.831532,0.083745
